In [1]:
%load_ext ElasticNotebook

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/small_bench/checkpoints/pre_cell_19.pickle

In [3]:
%%RecordEvent
%%time
### cell 19 ###

def add_gap_rows(essay):
    cols = ['discourse_start', 'discourse_end', 'discourse_type', 'gap_length', 'gap_end_length']
    # filter once, avoid .query
    df_base = train.loc[train['id'] == essay, cols].reset_index(drop=True)
    print(df_base)
    if df_base.empty:
        return df_base
    # identify gap rows between existing segments (i >= 1)
    mask = df_base['gap_length'] > 0
    mask.iloc[0] = False
    gap_idxs = mask[mask].index
    if not gap_idxs.empty:
        starts = df_base['discourse_end'].shift(1).loc[gap_idxs] + 1
        ends = df_base['discourse_start'].loc[gap_idxs] - 1
        df_mid = pd.DataFrame({
            'discourse_start': starts.values,
            'discourse_end': ends.values,
            'discourse_type': ['Nothing'] * len(gap_idxs),
            'gap_length': [np.nan] * len(gap_idxs),
            'gap_end_length': [np.nan] * len(gap_idxs)
        })
    else:
        df_mid = pd.DataFrame(columns=cols)
    # final gap after last segment
    if df_base['gap_end_length'].iloc[-1] > 0:
        final_start = df_base['discourse_end'].iloc[-1] + 1
        final_end = final_start + df_base['gap_end_length'].iloc[-1]
        df_final = pd.DataFrame({
            'discourse_start': [final_start],
            'discourse_end': [final_end],
            'discourse_type': ['Nothing'],
            'gap_length': [np.nan],
            'gap_end_length': [np.nan]
        })
    else:
        df_final = pd.DataFrame(columns=cols)
    # combine and sort once
    df_out = pd.concat([df_base, df_mid, df_final], ignore_index=True)
    return df_out.sort_values('discourse_start').reset_index(drop=True)


CPU times: user 4 µs, sys: 1e+03 ns, total: 5 µs
Wall time: 7.15 µs


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_19_try_0.pickle